# Ch 22. DPO と最新アラインメント — 解答

> このノートブックは 5 問すべての再現可能な解答を提供する。出力は消去済みで、コードセルは CPU のみの環境で上から順に実行できる。


## 問題 1

**問題**: DPO 損失で $\beta=0.01,0.1,1.0$ を比較し、効果を説明せよ。

### 解答

DPO logit は $\beta[(\log\pi_w-\log\pi_{ref,w})-(\log\pi_l-\log\pi_{ref,l})]$ である。同じ margin では大きな $\beta$ が sigmoid を飽和させ、鋭い選好と小さな有効勾配を生み得る。損失と勾配を比較する。


In [ ]:
import numpy as np

margin=2.0; results={}
for beta in (.01,.1,1.0):
    logit=beta*margin; loss=float(np.logaddexp(0,-logit)); gradient=float(-beta/(1+np.exp(logit)))
    results[beta]={"loss":loss,"d_loss_d_margin":gradient}
assert results[1.0]["loss"] < results[.1]["loss"] < results[.01]["loss"]
print({str(k):{m:round(v,6) for m,v in r.items()} for k,r in results.items()})


## 問題 2

**問題**: Bradley–Terry モデルが「選好対」をどのようにモデル化するか説明せよ。

### 解答

Bradley–Terry は潜在効用差を logit とし、$P(y_w\succ y_l)=\sigma(r_w-r_l)$ と置く。両順序の確率和は 1 で、両スコアに共通定数を加えても変わらない。


In [ ]:
import numpy as np

chosen=np.array([-1.0,0.0,2.0]); rejected=np.array([0.5,0.0,-1.0]); gaps=chosen-rejected
prob=1/(1+np.exp(-gaps)); reverse=1/(1+np.exp(gaps))
assert np.allclose(prob+reverse,1) and np.isclose(prob[1],.5)
print([{"utility_gap":float(g),"P(chosen preferred)":round(float(p),4)} for g,p in zip(gaps,prob)])


## 問題 3

**問題**: DPO と PPO のメモリ使用量を比較せよ。

### 解答

PPO は通常 policy・reference・reward・value の 4 モデルと rollout buffer を要する。DPO は policy と reference の chosen/rejected 順伝播が中心である。厳密な byte 数は optimizer 等に依存するが、同精度でモデル状態の下限は概ね半分になる。


In [ ]:
# Explicit memory ledger in model-size units; batch buffers are separately measured assumptions.
ppo={"policy":1.0,"reference":1.0,"reward":1.0,"value":1.0,"rollouts":0.30}
dpo={"policy":1.0,"reference":1.0,"paired_batch":0.10}
totals={"PPO":sum(ppo.values()),"DPO":sum(dpo.values())}
assert totals["DPO"] < totals["PPO"] and set(ppo)-set(dpo)=={"reward","value","rollouts"}
print({"components":{"PPO":ppo,"DPO":dpo},"totals":totals,"DPO_fraction":round(totals["DPO"]/totals["PPO"],3)})


## 問題 4

**問題**: KTO が選好対なしで学習できる理由を説明せよ。

### 解答

KTO は各応答を desirable/undesirable と標識し、reference に対する対数確率比から得る価値を prospect theory 型非対称損失で最適化する。各ラベル応答単独で損失項を作れるため同一 prompt の明示的対は不要である。


In [ ]:
import numpy as np

# Independent labeled examples contribute without constructing chosen/rejected pairs.
log_ratio=np.array([.7,-.4,.2,-.8]); desirable=np.array([1,0,1,0],dtype=bool)
signed_value=np.where(desirable,log_ratio,-log_ratio)
loss=np.logaddexp(0,-signed_value)
assert loss.shape==(4,) and loss[0] < loss[2] and loss[3] < loss[1]
print([{"label":"desirable" if y else "undesirable","log_ratio":float(r),"loss":round(float(l),4)}
       for y,r,l in zip(desirable,log_ratio,loss)])


## 問題 5

**問題**: 報酬–方策関係式 $r = \beta \log \frac{\pi^*}{\pi_{\text{ref}}} + \beta \log Z$ を導出せよ。

### 解答

各 $x$ の KL 正則化目的を Lagrange 乗数で解くと $\pi^*(y|x)\propto\pi_{ref}(y|x)e^{r(x,y)/\beta}$。正規化定数 $Z(x)$ を入れ対数を取り整理すると式を得て、応答差では $\beta\log Z(x)$ が相殺する。


In [ ]:
import numpy as np

reward=np.array([1.2,-.3,.7]); beta=.2; reference=np.array([.2,.5,.3])
unnormalized=reference*np.exp(reward/beta); policy=unnormalized/unnormalized.sum(); Z=unnormalized.sum()
reconstructed=beta*np.log(policy/reference)+beta*np.log(Z)
max_error=float(np.max(np.abs(reconstructed-reward)))
assert np.allclose(reconstructed,reward,atol=1e-12) and np.isclose(policy.sum(),1)
print({"reference":reference.tolist(),"optimal_policy":policy.round(6).tolist(),"partition_Z":round(float(Z),6),"max_reconstruction_error":max_error})
